# **Step 1:**
## **installing odf**

In [ ]:
!pip install odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160673 sha256=88974df7c76a3382789433be5e60c9c91779d2dd36cf1fffa8ac33a84f89bd8b
  Stored in directory: /root/.cache/pip/wheels/36/5d/63/8243a7ee78fff0f944d638fd0e66d7278888f5e2285d7346b6
Successfully built odfpy


# **Step 2:**
**Converting CSV file to ODS**

In [ ]:
# importing imp things
from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

# Converting CSV to ODS
import pandas as pd
df = pd.read_csv('/content/avocado.csv', index_col=0)
df.to_excel('avocado.ods', engine='odf', index=False)
print('Conversion completed!')

Conversion completed!


# **Step 3:**
**Loading the ODS file and printing sheet names and column names**

In [ ]:
from odf import table
# loading the ODS file
file_name = 'avocado.ods'
doc = load(file_name)

# Extracting the 1st sheet
sheets = doc.getElementsByType(Table)
sheet = sheets[0]
sheet_name = sheet.getAttribute('name')
rows = sheet.getElementsByType(TableRow)

# Extract column names from sheet 1
header_row = rows[0]
headers = [
    teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
    for cell in header_row.getElementsByType(TableCell)
    ]

print(f'Sheet name: {sheet_name}')
print(f'Column names: {headers}')


Sheet name: Sheet1
Column names: ['Date', 'AveragePrice', 'Total Volume', '4046', '4225', '4770', 'Total Bags', 'Small Bags', 'Large Bags', 'XLarge Bags', 'type', 'year', 'region']


# **Step 4:**
 **Display Rows**

In [ ]:
rows = sheet.getElementsByType(TableRow)
for i, row in enumerate(rows[1:6], 1):
  cells = row.getElementsByType(TableCell)
  values = [
      teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
      for cell in cells
  ]
  print (f'Row {i}: {values}')

Row 1: ['2015-12-27', '1.33', '64236.62', '1036.74', '54454.85', '48.16', '8696.87', '8603.62', '93.25', '', 'conventional', '2015', 'Albany']
Row 2: ['2015-12-20', '1.35', '54876.98', '674.28', '44638.81', '58.33', '9505.56', '9408.07', '97.49', '', 'conventional', '2015', 'Albany']
Row 3: ['2015-12-13', '0.93', '118220.22', '794.7', '109149.67', '130.5', '8145.35', '8042.21', '103.14', '', 'conventional', '2015', 'Albany']
Row 4: ['2015-12-06', '1.08', '78992.15', '1132.0', '71976.41', '72.58', '5811.16', '5677.4', '133.76', '', 'conventional', '2015', 'Albany']
Row 5: ['2015-11-29', '1.28', '51039.6', '941.48', '43838.39', '75.78', '6183.95', '5986.26', '197.69', '', 'conventional', '2015', 'Albany']


# **Step 5:**
**Cleaning the Data**

In [ ]:
# Cleaning headers
clean_headers = [h.strip().lower().replace(' ', '_') for h in headers]

# Cleaning data
clean_data = []
for row in rows[1:]:
  cells = row.getElementsByType(TableCell)
  if not cells or all(len(cell.getElementsByType(P)) == 0 for cell in cells):
    continue
  values = [
      teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
      for cell in cells
  ]

  # Pad values if row is short
  if len(values) < len(clean_headers):
    values += [''] * (len(clean_headers) - len(values))
    # covert empty string to none
  values = [v if v != '' else None for v in values]
  record = dict(zip(clean_headers, values))
  clean_data.append(record)
print(f"Loaded and cleaned {len(clean_data)} records.")
print("Sample cleaned record:")
print(clean_data[0])


Loaded and cleaned 18249 records.
Sample cleaned record:
{'date': '2015-12-27', 'averageprice': '1.33', 'total_volume': '64236.62', '4046': '1036.74', '4225': '54454.85', '4770': '48.16', 'total_bags': '8696.87', 'small_bags': '8603.62', 'large_bags': '93.25', 'xlarge_bags': None, 'type': 'conventional', 'year': '2015', 'region': 'Albany'}


# **Step 6:**
**Adding Clean Data to ODS file**

In [ ]:
from odf.opendocument import OpenDocumentSpreadsheet
from odf.table import Table, TableRow, TableCell
from odf.text import P

cleaned_file = 'cleaned_avocado.ods'
cleaned_doc = OpenDocumentSpreadsheet()
cleaned_table = Table(name= 'CleanedAvocado')

# Adding header row
header_row = TableRow()
for h in clean_headers:
  cell = TableCell()
  cell.addElement(P(text=h))
  header_row.addElement(cell)
cleaned_table.addElement(header_row)

# Adding Clean Data
for rec in clean_data:
  row = TableRow()
  for h in clean_headers:
    val = rec.get(h, '')
    cell = TableCell()
    cell.addElement(P(text=str(val) if val is not None else ''))
    row.addElement(cell)
  cleaned_table.addElement(row)
# Finally save everything in cleaned_file
cleaned_doc.spreadsheet.addElement(cleaned_table)
cleaned_doc.save(cleaned_file)
print(f'Cleaned data saved as {cleaned_file}')

Cleaned data saved as cleaned_avocado.ods


# **Step 7: **
**Doing Questions**

**Q1: Average price of Avocado in each region**

In [ ]:
from collections import defaultdict

region_prices = defaultdict(list)
for rec in clean_data:
  region = rec.get('region')
  try:
    prices = float(rec.get('averageprice', 0) or 0)
    if region and prices > 0:
        region_prices[region].append(prices)
  except Exception:
    continue

  # Finding the average
print('Average price by region is: ')
for region, prices in region_prices.items():
  avg = sum(prices) / len(prices) if prices else 0
  print(f'{region}: {avg:.2f}')



Average price by region is: 
Albany: 1.56
Atlanta: 1.34
BaltimoreWashington: 1.53
Boise: 1.35
Boston: 1.53
BuffaloRochester: 1.52
California: 1.40
Charlotte: 1.61
Chicago: 1.56
CincinnatiDayton: 1.21
Columbus: 1.25
DallasFtWorth: 1.09
Denver: 1.22
Detroit: 1.28
GrandRapids: 1.50
GreatLakes: 1.34
HarrisburgScranton: 1.51
HartfordSpringfield: 1.82
Houston: 1.05
Indianapolis: 1.31
Jacksonville: 1.51
LasVegas: 1.38
LosAngeles: 1.22
Louisville: 1.29
MiamiFtLauderdale: 1.43
Midsouth: 1.40
Nashville: 1.21
NewOrleansMobile: 1.30
NewYork: 1.73
Northeast: 1.60
NorthernNewEngland: 1.48
Orlando: 1.51
Philadelphia: 1.63
PhoenixTucson: 1.22
Pittsburgh: 1.36
Plains: 1.44
Portland: 1.32
RaleighGreensboro: 1.56
RichmondNorfolk: 1.29
Roanoke: 1.25
Sacramento: 1.62
SanDiego: 1.40
SanFrancisco: 1.80
Seattle: 1.44
SouthCarolina: 1.40
SouthCentral: 1.10
Southeast: 1.40
Spokane: 1.45
StLouis: 1.43
Syracuse: 1.52
Tampa: 1.41
TotalUS: 1.32
West: 1.27
WestTexNewMexico: 1.26


**Q2: Which type of avocado (conventional/organic) has the highest average total volume sold?**

In [ ]:
type_volumes = defaultdict(list)

for rec in clean_data:
  avocado_type = rec.get('type')
  try:
    volume = float(rec.get('total_volume', 0) or 0)
    if avocado_type and volume > 0:
      type_volumes[avocado_type].append(volume)
  except Exception:
    continue

  # Calculating the average
max_type = None
max_avg = 0
for avocado_type, volumes in type_volumes.items():
    avg = sum(volumes) / len(volumes) if volumes else 0
    print(f"{avocado_type}: {avg:.2f}")
    if avg > max_avg:
      max_avg = avg
      max_type = avocado_type
print(f'Type with highest average total volume: {max_type}: {max_avg:.2f}')


conventional: 1653212.90
organic: 47811.21
Type with highest average total volume: conventional: 1653212.90


**Q3: What is the average price of Avocadoes for each year?**

In [ ]:
year_price = defaultdict(list)

for rec in clean_data:
  year = rec.get('year')
  try:
    prices = float(rec.get('averageprice', 0) or 0)
    if  year and prices > 0:
      year_price[year].append(prices)
  except Exception:
    continue

  # Calculating average price for years
for year, price in year_price.items():
    avg = sum(price) / len(price) if price else ''
    print(f'{year}: {avg:.2f}')

2015: 1.38
2016: 1.34
2017: 1.52
2018: 1.35


# **Task 1**

**Find the top 10 regions with the highest average price**

In [ ]:
from os import name
from odf.opendocument import OpenDocumentSpreadsheet
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf.style import Style, TableCellProperties
from collections import defaultdict

# Calculating the average
region_prices = defaultdict(list)
for rec in clean_data:
  region = rec.get('region')
  try:
    prices = float(rec.get('averageprice', 0) or 0)
    if region and prices > 0:
        region_prices[region].append(prices)
  except Exception:
    continue

  # Finding the average
region_avg = {region: sum(prices) / len(prices) for region, prices in region_prices.items() if prices}
top_10_regions = sorted(region_avg.items(), key=lambda x: x[1], reverse=True)[:10]
top_10_regions_ranks = {region: rank+1 for rank, (region, _) in enumerate(top_10_regions)}

# Add the top 10 rank region
enhanced_data = []
for rec in clean_data:
  region = rec.get('region')
  rank = top_10_regions_ranks.get(region)
  enhanced_record = rec.copy()
  enhanced_record['top10_region_price_rank'] = rank if rank is not None else ''
  enhanced_data.append(enhanced_record)

# Create ODS file with two sheets and color-coded cells
doc1 = OpenDocumentSpreadsheet()
green_colour_cell = Style(name='GreenCell', family='table-cell')
green_colour_cell.addElement(TableCellProperties(backgroundcolor='#CCFFCC'))
yellow_colour_cell = Style(name='YellowCell', family='table-cell')
yellow_colour_cell.addElement(TableCellProperties(backgroundcolor='#FFFF99'))
red_colour_cell = Style(name='RedCell', family='table-cell')
red_colour_cell.addElement(TableCellProperties(backgroundcolor='#FFCCCC'))
doc1.automaticstyles.addElement(green_colour_cell)
doc1.automaticstyles.addElement(yellow_colour_cell)
doc1.automaticstyles.addElement(red_colour_cell)

# Sheet 1: Cleaned data + top 10 region rank, with color coding for the rank column
table1 = Table(name='CleanedAvocadoWithTop10RegionPriceRank')
header_row1 = TableRow()
for h in clean_headers + ['top10_region_price_rank']:
  cell = TableCell()
  cell.addElement(P(text=h))
  header_row1.addElement(cell)
table1.addElement(header_row1)
for rec in enhanced_data:
  row = TableRow()
  for h in clean_headers:
    val = rec.get(h, '')
    cell = TableCell()
    cell.addElement(P(text=str(val) if val is not None else ''))
    row.addElement(cell)
  rank = rec.get('top10_region_price_rank')
  style = None
  if str(rank) in ['1', '2', '3']:
    style = 'GreenCell'
  elif str(rank) in ['4', '5', '6', '7']:
    style = 'YellowCell'
  elif str(rank) in ['8', '9', '10']:
    style = "RedCell"
  cell = TableCell(stylename= style) if style else TableCell()
  cell.addElement(P(text=str(rank) if rank is not None else ''))
  row.addElement(cell)
  table1.addElement(row)
doc1.spreadsheet.addElement(table1)

# Sheet 2: # Sheet 2: Top 10 regions by price, color-coded cells and with rank
table2 = Table(name='Top10RegionPrice')
header_row2 = TableRow()
for col in ['Rank', 'Region', 'Average Price']:
  cell = TableCell()
  cell.addElement(P(text=col))
  header_row2.addElement(cell)
table2.addElement(header_row2)
for idx, (region, avg_price) in enumerate(top_10_regions):
  if idx < 3:
    cell_style = 'GreenCell'
  elif idx < 7:
    cell_style = 'YellowCell'
  else:
    cell_style = 'RedCell'
  trow = TableRow()
  for value in [str(idx+1), str(region), str(round(avg_price, 3))]:
    cell = TableCell(stylename=cell_style)
    cell.addElement(P(text=value))
    trow.addElement(cell)
  table2.addElement(trow)
doc1.spreadsheet.addElement(table2)
task1_file = 'task1_top10_region_price.ods'
doc1.save(task1_file)
print(f'Task 1 file saved as {task1_file}')

Task 1 file saved as task1_top10_region_price.ods


# **Verifier**

In [ ]:
from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf.style import TableCellProperties
from collections import defaultdict

def get_sheet_names(odf_doc):
  return [s.getAttribute('name') for s in odf_doc.getElementsByType(Table)]
def get_headers(sheet):
  header_row = sheet.getElementsByType(TableRow)[0]
  return [P.firstChild.data.strip() for P in header_row.getElementsByType(P) if P.firstChild]
def get_data_rows(sheet):
  return sheet.getElementsByType(TableRow)[1:]
def get_cell_text(cell):
  ps = cell.getElementsByType(P)
  return ps[0].firstChild.data.strip() if ps and ps[0].firstChild else ''
def get_cell_style(cell):
  return cell.getAttribute('stylename')
def get_style_colormap(odf_doc):
  color_map = {}
  for style in odf_doc.automaticstyles.childNodes:
    if getattr(style, 'tagName', None) == 'style:style':
      props = style.getElementsByType(TableCellProperties)
      if props and props[0].getAttribute('backgroundcolor'):
        color_map[style.getAttribute('name')] = props[0].getAttribute('backgroundcolor')
  return color_map

def validate_task1_full(task1_file, cleaned_file):
    print('=== Task 1 Validation ===')
    cleaned_doc = load(cleaned_file)
    cleaned_sheet_names = get_sheet_names(cleaned_doc)
    print(f'Original File Names: {cleaned_sheet_names}')
    cleaned_sheet = cleaned_doc.getElementsByType(Table)[0]
    cleaned_headers = get_headers(cleaned_sheet)
    print(f'Original Header: {cleaned_headers}')

    task1_doc = load(task1_file)
    task1_sheet_names = get_sheet_names(task1_doc)
    print(f'Task 1 Sheet Name: {task1_sheet_names}')

    for name in cleaned_sheet_names:
      print(f'File {name} is present in output: {name in task1_sheet_names}')
    table = [t for t in task1_doc.getElementsByType(Table) if t.getAttribute('name') == 'Top10RegionPrice']
    if not table:
      print('  Sheet \'Top10RegionPrice\' not found in output!')
      return
    table = table[0]
    headers = get_headers(table)
    print(f'Task 1 Headers: {headers}')
    task1_headers_match = headers == ['Rank', 'Region', 'Average Price']
    print(f'Header names match: {task1_headers_match}')
    print(f'Header length match: {len(headers) == 3}')
    # Recompute top 10 by region from cleaned data
    region_prices = defaultdict(list)
    for row in get_data_rows(cleaned_sheet):
        cells = row.getElementsByType(TableCell)
        row_dict = dict(zip(cleaned_headers, [get_cell_text(cell) for cell in cells]))
        try:
            region = row_dict.get('region')
            price = float(row_dict.get('averageprice', 0) or 0)
            if region and price > 0:
                region_prices[region].append(price)
        except Exception:
            continue
    region_avg = {region: round(sum(prices)/len(prices), 3) for region, prices in region_prices.items() if prices}
    expected = sorted(region_avg.items(), key=lambda x: x[1], reverse=True)[:10]
    color_map = get_style_colormap(task1_doc)
    value_errors = 0
    color_errors = 0

    actual_rows = list(get_data_rows(table))
    n = min(len(expected), len(actual_rows))
    for i in range(n):
        row = actual_rows[i]
        cells = row.getElementsByType(TableCell)
        values = [get_cell_text(cell) for cell in cells]
        exp_rank = str(i+1)
        exp_region, exp_price = expected[i]
        if values[0] != exp_rank or values[1] != exp_region or abs(float(values[2]) - exp_price) > 0.01:
            print(f'  Row {i+1}: value mismatch (expected {exp_rank}, {exp_region}, {exp_price}, got {values})')
            value_errors += 1
        for cell in cells:
            style = get_cell_style(cell)
            actual_color = color_map.get(style)
            if i < 3:
                expected_color = "#CCFFCC"
            elif i < 7:
                expected_color = "#FFFF99"
            else:
                expected_color = "#FFCCCC"
            if actual_color != expected_color:
                print(f'  Row {i+1}: color mismatch (expected {expected_color}, got {actual_color})')
                color_errors += 1
    if len(actual_rows) != len(expected):
        print(f'WARNING: Number of rows in Top10RegionPrice sheet ({len(actual_rows)}) does not match expected ({len(expected)})')

    print(f'- Average value validation errors: {value_errors}')
    print(f'- Color validation errors: {color_errors}')

# Example usage:
validate_task1_full('task1_top10_region_price.ods', 'cleaned_avocado.ods')

=== Task 1 Validation ===
Original File Names: ['CleanedAvocado']
Original Header: ['date', 'averageprice', 'total_volume', '4046', '4225', '4770', 'total_bags', 'small_bags', 'large_bags', 'xlarge_bags', 'type', 'year', 'region']
Task 1 Sheet Name: ['CleanedAvocadoWithTop10RegionPriceRank', 'Top10RegionPrice']
File CleanedAvocado is present in output: False
Task 1 Headers: ['Rank', 'Region', 'Average Price']
Header names match: True
Header length match: True
- Average value validation errors: 0
- Color validation errors: 0
